In [ ]:
"""
exp02 Step 01: development replay (PROTOCOL.md version 2.0)

Re-run of protocol 1.0 under the current rules, not new evidence. Scores the 18 candidates on the development
quarters (fresh windows), then replays the prior-only path through the development end. The holdout is not read.
Logs one validation trial per candidate and period, and one summary, with note "re-run under new rules".
Prerequisite: raw SPY bars and a registered holdout window. Outputs: store04_experiments/exp02_stop_reentry_v2/step01_development_data/.
"""
None

In [ ]:
# IMPORT GENERAL PACKAGES
import pandas as pd
import numpy as np
# IMPORT THE SHARED CONFIGURATION, THE EXPERIMENT CONFIGURATION AND THE PATHS
from so import config, paths
from experiments.exp02_stop_reentry import config as exp_config
# IMPORT THE DEVELOPMENT REPORT
from experiments.exp02_stop_reentry.report import run_development_dict
# IMPORT LOCAL FILE MANAGEMENT FUNCTIONS
from so.core.local_file_management import write_csv_file_to_path
# IMPORT TRIAL LOG FUNCTIONS
from so.core.trial_log import get_config_hash_str, log_trial_pdf, log_trial_dict, get_trial_count_pdf, print_previous_trial_warning

In [ ]:
# DISPLAY THE EXPERIMENT, THE PROTOCOL VERSION AND THE CONFIGURATION HASH
print(f"Experiment:\t{exp_config.EXPERIMENT_NAME}\tProtocol:\t{exp_config.PROTOCOL_VERSION}\tConfig Hash:\t{get_config_hash_str(exp_config)}")
# DEFINE THE OUTPUT FOLDER OF THIS STEP
output_path_str = paths.get_experiment_data_path_str(exp_config.EXPERIMENT_NAME, "step01_development_data")
print(f"Output folder:\t{output_path_str}")
# WARN WHEN THIS STAGE ALREADY HAS TRIALS (A RE-RUN ADDS ROWS)
previous_trial_count = print_previous_trial_warning(exp_config, "validation")

In [ ]:
# RUN THE DEVELOPMENT GRID, THE PRIOR-ONLY PATH AND THE BASELINES (ENDS AT THE DEVELOPMENT END)
result_dict = run_development_dict()
span_dict = result_dict["span_dict"]
# DISPLAY THE BOUNDARY AND THE LAST PRICE DATE THAT WAS LOADED (THE WINDOW ITSELF IS NOT LOADED)
print(f"Development end:\t{span_dict['development_end']}")
print(f"Last price date loaded before the cut:\t{span_dict['last_price_date']}")
print(f"Periods:\t{len(result_dict['period_pdf'])}\tfrom {result_dict['period_pdf']['period_start'].iloc[0]} to {result_dict['period_pdf']['period_end'].iloc[-1]}")

In [ ]:
# THE OPTIMISTIC CANDIDATE IS THE BEST CHAIN OF FRESH WINDOWS AFTER THE FACT
optimistic_dict = result_dict["optimistic_dict"]
print("Optimistic (best candidate after the fact):")
print(f"\t{optimistic_dict['candidate']}\t{optimistic_dict['total_return']:.2%} vs buy-and-hold {optimistic_dict['buy_hold_total_return']:.2%}")
# THE PRIOR-ONLY PATH IS THE HONEST ESTIMATE
path_dict = result_dict["path_dict"]
buy_hold_return = result_dict["buy_hold_dict"]["metric_dict"]["total_return"]
path_return = path_dict["metric_dict"]["total_return"]
print("Prior-only path:")
print(f"\t{path_return:.2%} vs buy-and-hold {buy_hold_return:.2%}")
print(f"\ttime in the market:\t{1.0 - path_dict['metric_dict']['cash_session_pct']:.1%}")
print(f"\tstops:\t{path_dict['metric_dict']['stop_count']}\tforced re-entries:\t{path_dict['metric_dict']['forced_reentry_count']}")
# UNINFORMED BASELINES ON THE SAME SPAN
baseline_dict = result_dict["baseline_dict"]
for name_str in ["fixed_delay_1", "fixed_delay_5", "fixed_delay_20", "random_reentry_median", "trend_ma200"]:
    print(f"\t{name_str}:\t{baseline_dict[name_str]:.2%}")
random_beaten = float(np.mean(np.array(baseline_dict["random_return_list"]) < path_return))
print(f"\trandom runs beaten:\t{random_beaten:.0%} of {len(baseline_dict['random_return_list'])}")
# MINIMUM DETECTABLE EFFECT AND THE LOG-EXCESS INTERVAL
summary_dict = result_dict["summary_dict"]
log_excess_dict = result_dict["log_excess_dict"]
print(f"MDE per quarter:\t{summary_dict['mde_per_window']:.2%}\tper year:\t{summary_dict['mde_per_year']:.2%}")
print(f"Log excess:\t{log_excess_dict['annualized_log_excess']:.2%}  [{log_excess_dict['ci_low']:.2%}, {log_excess_dict['ci_high']:.2%}]")
# TURNOVER, SWITCHES, REFITS
activity_dict = result_dict["activity_dict"]
refit_dict = result_dict["refit_dict"]
print(f"Turnover per year:\t{activity_dict['turnover_per_year']:.2f}\ttrades:\t{activity_dict['trade_count']}\tswitches:\t{activity_dict['candidate_switch_count']}")
print(f"Refits used:\trows {refit_dict['mean_rows_used']:.1f}\tpurged share {refit_dict['mean_share_purged']:.1%}\tprediction move {refit_dict['mean_prediction_move']:.4f}")
print(f"Verdict:\t{result_dict['verdict_str']}\tbeats buy-and-hold {result_dict['beats_buy_hold_bool']}\tbeats fixed delays {result_dict['beats_delay_bool']}")
print("Success flags:", {key: result_dict["success_dict"][key] for key in ["primary_success", "secondary_success", "comparable_return", "all_baselines_beaten"]})

In [ ]:
# SAVE THE TABLES
write_csv_file_to_path(result_dict["score_pdf"], f"{output_path_str}score_data.csv", "W", alert_in=False)
write_csv_file_to_path(result_dict["selection_pdf"], f"{output_path_str}selection_data.csv", "W", alert_in=False)
write_csv_file_to_path(result_dict["period_pdf"], f"{output_path_str}period_data.csv", "W", alert_in=False)
write_csv_file_to_path(path_dict["daily_equity_pdf"], f"{output_path_str}prior_only_daily_equity_data.csv", "W", alert_in=False)
write_csv_file_to_path(result_dict["performance_pdf"], f"{output_path_str}performance_over_time_data.csv", "W", alert_in=False)
write_csv_file_to_path(result_dict["quarter_pdf"], f"{output_path_str}performance_quarter_data.csv", "W", alert_in=False)
write_csv_file_to_path(path_dict["refit_log_pdf"], f"{output_path_str}refit_log_data.csv", "W", alert_in=False)
summary_row_dict = {"verdict": result_dict["verdict_str"], "path_return": path_return, "buy_hold_return": buy_hold_return,
                    "optimistic_candidate": optimistic_dict["candidate"], "optimistic_return": optimistic_dict["total_return"],
                    "in_market_pct": 1.0 - path_dict["metric_dict"]["cash_session_pct"],
                    **{f"baseline_{key}": value for key, value in baseline_dict.items() if key != "random_return_list"},
                    **{f"mde_{key}": value for key, value in summary_dict.items() if np.isscalar(value)},
                    **{f"log_{key}": value for key, value in log_excess_dict.items() if np.isscalar(value)},
                    **{f"activity_{key}": value for key, value in activity_dict.items() if key != "candidate_time_share_dict"},
                    **refit_dict}
write_csv_file_to_path(pd.DataFrame([summary_row_dict]), f"{output_path_str}summary_data.csv", "W", alert_in=False)

In [ ]:
# LOG ONE VALIDATION TRIAL PER CANDIDATE AND PERIOD, THEN ONE SUMMARY
logged_score_pdf = log_trial_pdf(exp_config, "validation", result_dict["score_pdf"], ["candidate"],
                                 ["total_return", "buy_hold_total_return", "excess_return"], False, result_dict["fold_pdf"],
                                 "re-run under new rules")
summary_trial_dict = log_trial_dict(exp_config, "summary",
                                    {"path": "prior-only", "verdict": result_dict["verdict_str"], "optimistic_candidate": optimistic_dict["candidate"]},
                                    {"path_return": path_return, "buy_hold_return": buy_hold_return, "optimistic_return": optimistic_dict["total_return"],
                                     "in_market_pct": 1.0 - path_dict["metric_dict"]["cash_session_pct"]},
                                    False, note_str_in="re-run under new rules")
print(f"Logged validation rows:\t{len(logged_score_pdf)}")
print(f"Summary logged_ts:\t{summary_trial_dict['logged_ts']}")

In [ ]:
# DISPLAY THE TRIAL COUNT OF THIS EXPERIMENT
trial_count_pdf = get_trial_count_pdf()
experiment_count_pdf = trial_count_pdf[trial_count_pdf["experiment"] == exp_config.EXPERIMENT_NAME]
print(experiment_count_pdf.to_string(index=False))
print(f"Trials this step:\t{int(experiment_count_pdf['trial_count'].sum())}")